In [24]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import seaborn as sns
import matplotlib.pyplot as plt
import tempfile
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
import pickle

In [25]:
df = pd.read_csv('spam_data.csv')
print(df.columns)
print(df.head())

Index(['sms', 'label'], dtype='object')
                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                  Ok lar... Joking wif u oni...\r\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he goes to usf, he lives aro...      0


In [26]:
def extract_features(email):
    features = {}

    # Length of email
    features['email_len'] = len(email)

    # Number of uppercase words
    features['num_uppercase_words'] = len(re.findall(r'\b[A-Z]{2,}\b', email))

    # Number of exclamation marks
    features['num_exclamations'] = email.count('!')

    # Number of URLs
    features['num_links'] = len(re.findall(r'http[s]?://', email))

    # Presence of HTML
    features['has_html'] = int(bool(re.search(r'<[^>]+>', email)))

    # Presence of spammy_words
    spammy_words = [
    'hurry', 'limited', 'win', 'few', 'credited', 'cash prize', 'click',
    'congratulations', 'lottery', 'free', 'urgent', 'act now', 'exclusive',
    'guaranteed', 'winner', 'miracle', 'earn money', 'get paid', 'easy money',
    'no cost', 'risk-free', 'special promotion', 'buy now', 'order now',
    'instant access', 'double your income', 'extra cash', 'financial freedom',
    'lowest price', 'money back', 'offer expires', 'trial', 'unsecured credit',
    'weight loss', 'viagra', 'investment', 'billion', 'million dollars', 'high return','earn more', 'earn money from home','Free Entry']

    features['Word_presence'] = int(any(word.lower() in email.lower() for word in spammy_words))

    if features['Word_presence'] >=1:
             features['Word_presence'] =   features['Word_presence']*5
    return features

In [27]:

feature_rows = df['sms'].apply(extract_features)


features_df = pd.DataFrame(feature_rows.tolist())


final_df = pd.concat([features_df, df['label']], axis=1)



In [28]:
final_df.loc[(final_df['Word_presence'] >= 5) | (final_df['label'] == 1), 'label'] = 1


In [29]:
final_df

,email_len,num_uppercase_words,num_exclamations,num_links,has_html,Word_presence,label
0,113,0,0,0,0,0,0
1,31,0,0,0,0,0,0
2,157,2,0,0,0,5,1
3,51,0,0,0,0,0,0
4,63,0,0,0,0,0,0
...,...,...,...,...,...,...,...
5569,162,1,1,0,0,0,1
5570,38,0,0,0,0,0,0
5571,59,0,0,0,0,0,0
5572,127,0,0,0,0,5,1


In [30]:
final_df.to_csv("Extracted_Features_output_RG.csv", index=False)


In [31]:
mlflow.sklearn.autolog()
with mlflow.start_run():
    X = final_df.drop('label', axis=1)
    y = final_df['label']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



    model = RandomForestClassifier(n_estimators=100, random_state=42)


    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

#print("Accuracy:", accuracy_score(y_test, y_pred))

2025/07/05 17:56:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\DESKTOP-9NEE77\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/07/05 17:56:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\DESKTOP-9NEE77\anaconda3\Lib\site-pac

In [32]:
def predict_message(text):
    features = extract_features(text)
    input_df = pd.DataFrame([features])
    prediction = model.predict(input_df)[0]
    return "Spam" if prediction == 1 else "Ham"

In [33]:
sample_msg1 = "!Congratulations! You've won a $1000 gift card. Click here to claim now rahul!!!!!! Click on Link http://claimnow.com"
sample_msg2 = "Just a reminder, you can  about our meeting tomorrow. Please be on time."

print("Message 1:", predict_message(sample_msg1))
print("Message 2:", predict_message(sample_msg2))

2025/07/05 17:56:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\DESKTOP-9NEE77\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/07/05 17:56:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\DESKTOP-9NEE77\anaconda3\Lib\site-pac

Message 1: Spam
Message 2: Ham


In [34]:
with open("Spam_detection_RG.pkl", "wb") as file:
    pickle.dump(model,file)